In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import sklearn
import torch
from stml.io import load_clean_data, load_returns_panel

print(f"torch {torch.__version__}  |  numpy {np.__version__}  |  pandas {pd.__version__}  |  sklearn {sklearn.__version__}")

TICKER_NAMES = {
    'cl1s': 'Crude Oil (CL)',
    'es1s': 'S&P 500 e-mini (ES)',
    'fesx1s': 'Euro Stoxx 50 (FESX)',
    'gc1s': 'Gold (GC)',
    'hg1s': 'Copper (HG)',
    'ho1s': 'Heating Oil (HO)',
    'ng1s': 'Natural Gas (NG)',
    'nq1s': 'Nasdaq 100 (NQ)',
    'pl1s': 'Platinum (PL)',
    'rb1s': 'RBOB Gasoline (RB)',
    'si1s': 'Silver (SI)',
}

In [ ]:
DATA = Path('../..') / 'data'

ohlcv = pd.read_csv(DATA / 'ohlcv_data.csv')
signals = pd.read_csv(DATA / 'primary_signals.csv')

ohlcv_clean, signals_clean = load_clean_data()
ret_clean = load_returns_panel()

print(ohlcv.shape, signals.shape)
ohlcv.head()

In [ ]:
def get_returns_series(ohlcv: pd.DataFrame, instrument: str):
    df = ohlcv.set_index("date")
    df.index = pd.to_datetime(df.index) 
    df = df[df["instrument"] == instrument]
    df = np.log(df["close"]).diff().dropna()
    return df

# test
df = get_returns_series(ohlcv, "gc1s")

fig, ax = plt.subplots(figsize = (12,5))
(1 + df).cumprod().plot(ax = ax)

ax.xaxis.set_major_locator(mdates.YearLocator(2))  # tick every 2 years
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
    

In [ ]:
test_ohlcv = ohlcv_clean[ohlcv_clean["instrument"] == "gc1s"]
test_ohlc = test_ohlcv.set_index("date").drop(columns = ["instrument", "volume", "open_interest"])
test_ohlc

In [ ]:
test_ohlcv2 = ohlcv[ohlcv["instrument"] == "gc1s"]
test_ohlc2 = test_ohlcv2.set_index("date").drop(columns = ["instrument", "volume", "open_interest"])
test_ohlc2.loc["2020-01-03":]

In [ ]:
test_signals = signals_clean["gc1s"]
test_signals.index = signals_clean["date"]
test_signals

In [ ]:
# compute atr
def compute_atr(ohlc: pd.DataFrame, window = 14):

    high = ohlc["high"]
    low = ohlc["low"]

    prev_close = ohlc["close"].shift(1)

    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs()
    ], axis =1).max(axis = 1)
    
    # apply smoothing
    atr = tr.ewm(alpha = 1/window, adjust = False).mean() # set adjust = True to correct for early bias
    return atr

compute_atr(test_ohlc)

In [ ]:


# get the label for each signal series
def get_trp_barr(signals: pd.Series, ohlc: pd.DataFrame, span: int, k_tp: float, k_sl: float):
    """
    Parameters
    ---
    signals : pd.Series
        signal series starting from 2020-01-03 to 2022-06-30
    ohlcv : pd.DataFrame
        instrument specific ohlcv data truncated to match window of signal series
    span : int
        how many indices we are looking forward
    vol_tp : float
        vol used for deciding  tp threshold
    vol_sl : float
        vol used for deciding sl threshold
    """
    atrs = compute_atr(ohlc)
    labels = []
    for start_date, row in ohlc.iterrows():
        if start_date == ohlc.index[-span]:
            break
        close = row["close"]
        label = 0
        s = signals.loc[start_date]
        atr = atrs.loc[start_date]
        end_date = start_date + pd.Timedelta(days = span)
        window_prices = ohlc.loc[start_date:end_date, "close"]    
        if s == 0:
            labels.append(label)
            continue
    
        elif s == 1: 
            tp_threshold = close + k_tp * atr
            sl_threshold = close - k_sl * atr

            tp_flags = tp_threshold <= window_prices.values
            print(tp_flags)

            if tp_threshold <= window_prices.values.max():
                label = 1 # price crosses above tp_threshold within window
            elif sl_threshold >= window_prices.values.min():
                label = 0 # price crosses below sl_threshold within window
            else: # price neither crosses both barriers
                label = 0
            
    
        elif s == -1:
            tp_threshold = close - k_tp * atr
            sl_threshold = close+ k_sl * atr

            if tp_threshold >= window_prices.values.min():
                label = 1 # price crosses below tp_threshold within window
            elif sl_threshold <= window_prices.values.max():
                label = 0 # price crosses above sl_thresohld within window
            else:
                label = 0 #
        labels.append(label)
    return labels


        
get_trp_barr(test_signals, test_ohlc, 20, 2,2)